In [ ]:
import pandas as pd
import joblib
import sys
from sklearn.metrics import classification_report, accuracy_score

sys.path.append('../') 
import data_utils

print("======================================================")
print("KIỂM CHỨNG CHÉO MÔ HÌNH TRÊN BỘ DỮ LIỆU WELFAKE")
print("======================================================")

# 1. Tải Vectorizer và Model đã lưu từ ổ đĩa
tfidf_title_text = joblib.load('../models/tfidf_title_text.pkl')
lr_xai_model = joblib.load('../models/lr_xai_model.pkl')
print("Đã tải thành công bộ mã hóa TF-IDF và mô hình ISOT!")

# 2. Đọc dữ liệu WELFake
welfake_df = pd.read_csv('../data/raw/WELFake_Dataset.csv')
welfake_df = welfake_df.dropna(subset=['text', 'title']).reset_index(drop=True)

# Chạy thử nghiệm trên 5000 mẫu ngẫu nhiên cho nhẹ máy (có thể bỏ .sample nếu muốn chạy toàn bộ tập)
welfake_sample = welfake_df.sample(n=5000, random_state=42).reset_index(drop=True)

# 3. Tiền xử lý văn bản bằng hàm chuẩn của dự án
print("Đang tiền xử lý văn bản WELFake...")
welfake_sample['title_text'] = welfake_sample['title'] + " " + welfake_sample['text']
welfake_sample['cleaned_text'] = welfake_sample['title_text'].apply(data_utils.deep_clean_text).apply(data_utils.lemmatize_and_remove_stopwords)

# 4. Vector hóa bằng tfidf_title_text ĐÃ LOAD
print("Đang vector hóa dữ liệu...")
X_welfake_tfidf = tfidf_title_text.transform(welfake_sample['cleaned_text'])
y_welfake_true = welfake_sample['label'].values

# 5. Dự đoán
print("Đang chạy mô hình dự đoán...")
y_welfake_pred = lr_xai_model.predict(X_welfake_tfidf)

# 6. Xuất kết quả đánh giá
print("\n[KẾT QUẢ EXTERNAL VALIDATION TRÊN WELFAKE]")
print(f"Accuracy: {accuracy_score(y_welfake_true, y_welfake_pred):.4f}")
print("\nChi tiết (Classification Report):")
print(classification_report(y_welfake_true, y_welfake_pred, target_names=['True News (0)', 'Fake News (1)']))